In [ ]:
import io, sys, os, datetime, requests, json
from collections import defaultdict
import geemap
import numpy as np
import pandas as pd
import shapely
import geopandas as gpd
import ee

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
urbext_2020 = ee.FeatureCollection('projects/wri-datalab/SCL-Cities/urbanextents__bycountry_2020_v7')
urbext_data_2020 = geemap.ee_to_gdf(urbext_2020)

In [ ]:
#Using WorldPop only to get country codes
worldpop_ic = ee.ImageCollection("WorldPop/GP/100m/pop").filter(ee.Filter.eq('year', 2020))
popcodes = worldpop_ic.filter(ee.Filter.eq('year', 2020)).aggregate_array('country').getInfo()

In [ ]:
scale = 100
exc_res = defaultdict(list)
pop_res = defaultdict(list)
excpop_total = []
pop_total = []
out_dicts = []
for year in [2010, 2015, 2020, 2025]:
    print(year)
    vphrase = ['_v2', ''][int(year == 2025)]
    excdays = ee.Image(f'projects/wri-datalab/SCL-Cities/cams-eac4-exceedancedays_any{vphrase}_{year}')
    ghspop_img = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP").filter(ee.Filter.calendarRange(year, year, 'year')).select('population_count').first()
    urbext_2020 = ee.FeatureCollection(f'projects/wri-datalab/SCL-Cities/urbanextents__bycountry_2020_v7')
    urbext_data_2020 = geemap.ee_to_df(urbext_2020)
    for i in range(len(urbext_data_2020)):
        ua = urbext_data_year.iloc[i]
        ua_f = urbext_year.filter(ee.Filter.eq('city_ids', str(ua['city_ids'])))
        geom = ua_f.geometry()
        if ua['country'] in popcodes:
            try:
                local_excdays = excdays.reduceRegion(ee.Reducer.mean(), geom, scale, maxPixels=1e12).get('b1').getInfo()

                localpop_img = ghspop_img
                if localpop_img is None:
                    print(f"\nNo population image for {ua['country']} (i={i}), skipping")
                    continue

                localpop = localpop_img.reduceRegion(ee.Reducer.sum(), geom, scale, maxPixels=1e12).get('population_count').getInfo()

                exc_res[ua['country']].append(local_excdays)
                pop_res[ua['country']].append(localpop)
                if local_excdays:
                    excpop_total.append(local_excdays * localpop)
                    pop_total.append(localpop)
                print(i, end=' ')

            except ee.EEException as e:
                print(f"\nError on {ua['country']} (i={i}): {e}")
                continue
        else:
            print()
            print(ua['country'])
    out_dicts.append(
        {
            'country': 'global',
            'pop-averaged_exceedance_days_{0}'.format(year): sum(excpop_total) / sum(pop_total),
        }
    )


    country_list = list(set(exc_res.keys()))
    country_list.sort()
    for country in country_list:
        if len(country) > 2:
            excpop = 0
            for idx in range(len(exc_res[country])):
                if exc_res[country][idx]:
                    excpop += exc_res[country][idx] * pop_res[country][idx]
            out_dicts.append(
                {
                    'country': country,
                    'pop-averaged_exceedance_days_{0}'.format(year): excpop / sum(pop_res[country])
                }
            )
res_pd = pd.DataFrame(out_dicts)
res_pd.to_csv('CTY-10_air-pollutant-exceedance-days_ghsl_v2.csv')